# 17 — Coulomb Baseline: Physics Lower Bound for ESP Prediction

APBS solves the full Poisson-Boltzmann equation, accounting for:
- **Dielectric inhomogeneity**: ε_protein ≈ 2–4 vs ε_solvent ≈ 80
- **Ionic shielding**: Debye–Hückel screening from dissolved salt

Vacuum Coulomb ignores both effects and computes:

$$V(\mathbf{q}) = \sum_i \frac{q_i \cdot l_B^{\text{vac}}}{r_{iq}}$$

where $l_B^{\text{vac}} = e^2 / (4\pi\epsilon_0 k_B T) \approx 560$ Å at 298 K converts
charge/distance to kT/e. The $q_i$ are the PARSE partial charges already in every `.pqr`
file — exactly what APBS takes as input.

**Why this matters for the thesis:**
- The RBF reconstruction (r ≈ 0.983) is an analytic *ceiling* — reconstructs from APBS values directly.
- Vacuum Coulomb is a physics *floor* — same charge inputs, no PBE solver.
- The gap shows the GNN has learned to approximate solvation physics (dielectric + ionic), not just raw charge–distance.

Full-dataset results are loaded from `model_eval/baseline/coulomb/` (evaluated on the canonical
110-protein test split via `src.baseline_models.coulomb.evaluate --split test`).

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

PROJECT_ROOT = Path("../.." ).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import get_data_root
from src.utils.paths import ProteinPaths
from src.data.dataset import load_split_manifest

data_root  = Path(get_data_root())
THESIS_ROOT = data_root.parent
EVAL_DIR    = THESIS_ROOT / "model_eval" / "baseline" / "coulomb"
CKPT_ROOT   = THESIS_ROOT / "checkpoints"

pd.set_option("display.float_format", "{:.4f}".format)
print("data_root:", data_root)
print("eval_dir: ", EVAL_DIR)

## 1. Configuration

In [ ]:
# Bjerrum length in vacuum at 298 K [Å] — e² / (4πε₀ · kBT)
BJERRUM_VACUUM = 560.0
R_MIN          = 0.5    # clamp min distance to avoid singularities
CHUNK_SIZE     = 512    # query points per chunk to bound RAM

_, _, test_ids = load_split_manifest(data_root)
SAMPLE_ID = test_ids[0]
print(f"Sample protein : {SAMPLE_ID}")
print(f"Test proteins  : {len(test_ids)}")

## 2. Single-Protein Walkthrough

Load PQR (charges) + `_esp.npz` (query positions + APBS ground truth) for one protein.

In [ ]:
def parse_pqr(pqr_path: Path):
    """Read PQR → atom_pos (N,3) float32, atom_charges (N,) float32."""
    positions, charges = [], []
    with open(pqr_path) as f:
        for line in f:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            parts = line.split()
            if len(parts) < 5:
                continue
            x, y, z  = float(parts[-5]), float(parts[-4]), float(parts[-3])
            charge    = float(parts[-2])
            positions.append([x, y, z])
            charges.append(charge)
    return np.array(positions, dtype=np.float32), np.array(charges, dtype=np.float32)


def coulomb_potential(query_pos, atom_pos, atom_charges, chunk_size=CHUNK_SIZE):
    """V(q) [kT/e] = Σ_i  q_i · BJERRUM_VACUUM / r_iq"""
    N_q = len(query_pos)
    V   = np.zeros(N_q, dtype=np.float64)
    for start in range(0, N_q, chunk_size):
        end  = min(start + chunk_size, N_q)
        diff = query_pos[start:end, np.newaxis, :] - atom_pos[np.newaxis, :, :]
        r    = np.linalg.norm(diff, axis=-1)
        np.clip(r, R_MIN, None, out=r)
        V[start:end] = (atom_charges[np.newaxis, :] / r).sum(axis=-1) * BJERRUM_VACUUM
    return V.astype(np.float32)

In [ ]:
p = ProteinPaths(SAMPLE_ID, data_root)

atom_pos, atom_charges = parse_pqr(p.pqr_path)
print(f"Atoms          : {len(atom_pos):>6}")
print(f"Net charge [e] : {atom_charges.sum():>8.2f}")
print(f"Charge range   : [{atom_charges.min():.3f}, {atom_charges.max():.3f}]")

esp_data  = np.load(p.esp_path)
verts     = esp_data["verts"]
esp_verts = esp_data["esp_verts"]
query_idx = esp_data["query_idx"]
query_pos  = verts[query_idx]
true_esp   = esp_verts[query_idx]

print(f"\nMesh vertices  : {len(verts):>6}")
print(f"Query points   : {len(query_idx):>6}  ({100*len(query_idx)/len(verts):.1f}% of mesh)")
print(f"APBS ESP range : [{true_esp.min():.2f}, {true_esp.max():.2f}] kT/e")

In [ ]:
coulomb_esp = coulomb_potential(query_pos, atom_pos, atom_charges)
r, _   = pearsonr(coulomb_esp.astype(float), true_esp.astype(float))
rmse   = float(np.sqrt(np.mean((coulomb_esp - true_esp) ** 2)))

print(f"Coulomb baseline — {SAMPLE_ID}")
print(f"  Pearson r        : {r:.4f}")
print(f"  RMSE             : {rmse:.2f} kT/e")
print(f"  ESP range (pred) : [{coulomb_esp.min():.2f}, {coulomb_esp.max():.2f}]")
print(f"  ESP range (true) : [{true_esp.min():.2f}, {true_esp.max():.2f}]")
print()
print("Note: Coulomb RMSE is large because vacuum potential has no dielectric")
print("shielding — magnitudes are ~100× too large. Pearson r is still high")
print("because the spatial *pattern* of charges is correct.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"Coulomb baseline — {SAMPLE_ID}", fontsize=13, fontweight="bold")

ax = axes[0]
ax.scatter(true_esp, coulomb_esp, s=3, alpha=0.3, rasterized=True)
lo = min(true_esp.min(), coulomb_esp.min())
hi = max(true_esp.max(), coulomb_esp.max())
ax.plot([lo, hi], [lo, hi], "r--", lw=1, label="y = x")
ax.set_xlabel("APBS ESP (kT/e)")
ax.set_ylabel("Coulomb ESP (kT/e)")
ax.set_title(f"Parity  r={r:.3f}  RMSE={rmse:.1f} kT/e")
ax.legend(fontsize=9)

ax = axes[1]
residuals = coulomb_esp - true_esp
ax.hist(residuals, bins=60, color="steelblue", alpha=0.8, edgecolor="none")
ax.axvline(0, color="r", lw=1.5, linestyle="--")
ax.set_xlabel("Coulomb − APBS (kT/e)")
ax.set_ylabel("Count")
ax.set_title(f"Residuals   μ={residuals.mean():.1f}  σ={residuals.std():.1f}")

plt.tight_layout()
plt.show()

## 3. Test-Set Results

Pre-computed on the canonical 110-protein test split via:
```bash
python -m src.baseline_models.coulomb.evaluate --split test --workers 8
```

In [ ]:
df = pd.read_csv(EVAL_DIR / "coulomb_results.csv")
print(f"Loaded {len(df)} proteins from {EVAL_DIR / 'coulomb_results.csv'}")

print("\n=== TEST SET (110 proteins) ===")
print(f"  Pearson r   mean={df.pearson_r.mean():.4f}  "
      f"median={df.pearson_r.median():.4f}  std={df.pearson_r.std():.4f}")
print(f"  RMSE        mean={df.rmse.mean():.2f}  "
      f"median={df.rmse.median():.2f}  std={df.rmse.std():.2f}")
print(f"  MAE         mean={df.mae.mean():.2f}  "
      f"median={df.mae.median():.2f}  std={df.mae.std():.2f}")
print()
print("Note: RMSE/MAE are high because vacuum Coulomb has no dielectric shielding.")
print("Pearson r measures the spatial pattern — which is high because charge")
print("positions are correct. The GNN learns to correct the magnitude.")

display(df.sort_values("pearson_r").head(10).style.format({
    "pearson_r": "{:.4f}", "rmse": "{:.2f}", "mae": "{:.2f}"
}).set_caption("10 lowest r proteins"))

## 4. Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Coulomb Baseline — Test Set (110 proteins)",
             fontsize=13, fontweight="bold")

ax = axes[0]
ax.hist(df["pearson_r"], bins=20, color="tomato", alpha=0.85, edgecolor="white")
ax.axvline(df["pearson_r"].mean(), color="darkred", lw=2,
           label=f"μ = {df['pearson_r'].mean():.3f}")
ax.axvline(df["pearson_r"].median(), color="darkred", lw=1.5, ls="--",
           label=f"median = {df['pearson_r'].median():.3f}")
ax.set_xlabel("Pearson r", fontsize=11)
ax.set_ylabel("Proteins", fontsize=11)
ax.set_title("Pearson r distribution", fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
ax.hist(df["rmse"], bins=20, color="mediumorchid", alpha=0.85, edgecolor="white")
ax.axvline(df["rmse"].mean(), color="purple", lw=2,
           label=f"μ = {df['rmse'].mean():.1f}")
ax.axvline(df["rmse"].median(), color="purple", lw=1.5, ls="--",
           label=f"median = {df['rmse'].median():.1f}")
ax.set_xlabel("RMSE (kT/e)", fontsize=11)
ax.set_ylabel("Proteins", fontsize=11)
ax.set_title("RMSE distribution", fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## 5. Architecture Comparison Table

Coulomb sits between the geometry-only DGCNN and the chemistry DGCNN in Pearson r,
but its RMSE is ~100× worse than any trained model because it lacks dielectric shielding.

In [ ]:
def _load_r(path):
    try:
        return json.load(open(path))["global"]["pearson_r"]
    except Exception:
        return None

def _load_rmse(path):
    try:
        return json.load(open(path))["global"]["rmse"]
    except Exception:
        return None

EVAL_BASE = data_root.parent / "model_eval" / "baseline"

comparison = [
    {"Model": "DGCNN (geom-only)",
     "Pearson r": 0.0849, "RMSE (kT/e)": 3.98,
     "Notes": "surface xyz+normals only; eval data deleted — hardcoded"},
    {"Model": "Vacuum Coulomb",
     "Pearson r": _load_r(EVAL_DIR / "test_metrics.json"),
     "RMSE (kT/e)": _load_rmse(EVAL_DIR / "test_metrics.json"),
     "Notes": "physics floor — correct pattern, no dielectric shielding"},
    {"Model": "DGCNN (chem)",
     "Pearson r": _load_r(EVAL_BASE / "surface_dgcnn_chem" / "test_metrics.json"),
     "RMSE (kT/e)": _load_rmse(EVAL_BASE / "surface_dgcnn_chem" / "test_metrics.json"),
     "Notes": "surface + nearest-atom element & residue"},
    {"Model": "CNN3D",
     "Pearson r": _load_r(EVAL_BASE / "cnn3d" / "test_metrics.json"),
     "RMSE (kT/e)": _load_rmse(EVAL_BASE / "cnn3d" / "test_metrics.json"),
     "Notes": "3D U-Net on voxelised atom grid"},
    {"Model": "AttentionESPN 4/4/10",
     "Pearson r": _load_r(CKPT_ROOT / "attention_v2_qq10" / "test_metrics.json"),
     "RMSE (kT/e)": _load_rmse(CKPT_ROOT / "attention_v2_qq10" / "test_metrics.json"),
     "Notes": "heterogeneous GNN, multi-head cross-attention AQ"},
    {"Model": "DistanceESPN 10/10/10",
     "Pearson r": _load_r(CKPT_ROOT / "distance_v2_all10" / "test_metrics.json"),
     "RMSE (kT/e)": _load_rmse(CKPT_ROOT / "distance_v2_all10" / "test_metrics.json"),
     "Notes": "heterogeneous GNN, RBF mean-aggregation AQ"},
]

comp_df = pd.DataFrame(comparison)
display(comp_df.style.format({"Pearson r": "{:.4f}", "RMSE (kT/e)": "{:.2f}"}, na_rep="—")
        .set_caption("Architecture hierarchy — test set (110 proteins)"))

## 6. Interpretation

**Pearson r separates two regimes:**
- DGCNN geom-only (r=0.085): surface shape alone cannot predict ESP — chemistry is essential.
- Vacuum Coulomb (r=0.863): the *spatial pattern* of APBS ESP is almost entirely determined
  by charge position and sign. A physics formula with no free parameters achieves 0.863 r.
- Trained models (r=0.67–0.90): the gap between Coulomb and the GNN champions is small
  in Pearson r terms (+0.03), but the models provide something Coulomb cannot: correct
  **absolute magnitudes** after dielectric shielding.

**RMSE tells the real story:**
- Coulomb RMSE = 236 kT/e — magnitudes are ~100× too large without ε attenuation.
- GNN RMSE = 2.3–2.6 kT/e — the network has effectively learned to apply a spatially
  varying dielectric correction on top of the charge-distance signal.
- DGCNN (chem) RMSE = 3.1 — closer than CNN3D (2.6) in r, but RMSE is worse,
  suggesting the surface model captures pattern but under-estimates amplitudes.

**Conclusion:** Vacuum Coulomb defines the informational ceiling for pattern prediction
from charge positions alone. The GNN's job is not to rediscover Coulomb's law —
that signal is nearly saturated at r=0.863 — but to learn the *solvation correction*
that APBS applies through the Poisson-Boltzmann equation.